In [ ]:
## import libraries
import glob
import xarray as xr
import geopandas as gpd
import os
import re
from collections import defaultdict
import pandas as pd
from pathlib import Path
import gc

In [ ]:
source_dir = '/Volumes/T7/data/blanka/'
files = sorted(glob.glob(source_dir + 'C1/*.nc'))
files2 = sorted(glob.glob(source_dir + 'C2/*.nc'))

In [ ]:
def yield_files_by_month(file_list, date_pattern):

    # Dictionary to hold the groups: Key = (2024, 8), Value = [file1, file2...]
    grouped_files = defaultdict(list)

    for file_path in file_list:
        filename = os.path.basename(file_path)
        match = date_pattern.search(filename)
        
        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            grouped_files[(year, month)].append(file_path)
        else:
            print(f"Warning: Could not parse date from {filename}")

    for (year, month), files in grouped_files.items():
        yield (year, month), files

In [ ]:
def convert_nc_to_parquet(file_list, columns):
    """
    Reads a list of EarthCARE NC files, converts them to a GeoDataFrame,
    sorts by time and location, and saves as GeoParquet.
    """
    dataframes = []

    print(f"Processing {len(file_list)} files...")

    for file_path in file_list:
        try:
            # Open the NetCDF file
            with xr.open_dataset(file_path, decode_timedelta=True) as ds:

                # Convert to Pandas DataFrame
                subset = ds[columns]
                df = subset.to_dataframe().reset_index(drop=True)

                # Extract EarthCARE ID
                path_obj = Path(file_path)
                earthcare_id = path_obj.stem.split('_')[-1]                
                df['earthcare_id'] = earthcare_id

                # add to list
                dataframes.append(df)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")


    print("Concatenating data...")
    full_df = pd.concat(dataframes, ignore_index=True)

    del dataframes
    gc.collect()


    print("Converting to GeoDataFrame...")
    # Create the geometry column from coordinates
    geometry = gpd.points_from_xy(full_df['longitude'], full_df['latitude'])
    gdf = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")

    del full_df
    gc.collect()

    print("Sorting data...")
    gdf.sort_values(
        by=['geometry', 'group_time'], 
        inplace=True, 
        ignore_index=True
    )

    return gdf

### Collection 1

Convert and store the GLM files

In [ ]:
output_dir = '/Volumes/T7/data/blanka/C1_parquet/'
source = 'GLM'
date_pattern = re.compile(r"GLM_(\d{4})(\d{2})")
source_files = [f for f in files if source in f]
columns = ['group_id',
 'group_time',
 'latitude',
 'longitude',
 'radiance',
 'flash_id',
 'ec_time_diff',
 'parent_cluster_id',
 'cluster_id',
 'distance_from_nadir']

earthcare_ids = {}

for ((year, month), filelist) in yield_files_by_month(source_files, date_pattern):
    gdf = convert_nc_to_parquet(filelist, columns)
    gdf['source'] = source
    gdf.to_parquet(
        output_dir + f'{source}_{year}_{month}.parquet',
        engine='pyarrow',
        compression='zstd',
        write_covering_bbox=True, 
        schema_version='1.1.0'
    )
    for id in gdf['earthcare_id'].unique():
        earthcare_ids.setdefault(id, []).append(f'{source}_{year}_{month}.parquet')


Processing 381 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 286 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 248 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 253 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 232 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 275 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 262 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 304 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 368 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 362 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 344 files...
Concatenating data...
Converting to GeoDataFrame...
Sort

Convert and store the LI files

In [ ]:
output_dir = '/Volumes/T7/data/blanka/C1_parquet/'
source = 'LI'
date_pattern = re.compile(r"LI_(\d{4})(\d{2})")
source_files = [f for f in files if source in f]
columns = ['group_id',
 'group_time',
 'latitude',
 'longitude',
 'radiance',
 'flash_id',
 'parallax_corrected_lat',
 'parallax_corrected_lon',
 'ec_time_diff',
 'parent_cluster_id',
 'cluster_id',
 'distance_from_nadir']


for ((year, month), filelist) in yield_files_by_month(source_files, date_pattern):
    gdf = convert_nc_to_parquet(filelist,columns)
    gdf['source'] = source
    gdf.to_parquet(
        output_dir + f'{source}_{year}_{month}.parquet',
        engine='pyarrow',
        compression='zstd',
        write_covering_bbox=True, 
        schema_version='1.1.0'
    )
    
    for id in gdf['earthcare_id'].unique():
        earthcare_ids.setdefault(id, []).append(f'{source}_{year}_{month}.parquet')

Processing 305 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 244 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 201 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 221 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 220 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 290 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 267 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 313 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 337 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 340 files...
Concatenating data...
Converting to GeoDataFrame...
Sorting data...
Processing 286 files...
Concatenating data...
Converting to GeoDataFrame...
Sort

In [ ]:
earthcare_ids

{'01253H': ['GLM_2024_8.parquet'],
 '01042F': ['GLM_2024_8.parquet'],
 '01206H': ['GLM_2024_8.parquet'],
 '01214F': ['GLM_2024_8.parquet'],
 '01283H': ['GLM_2024_8.parquet'],
 '01112H': ['GLM_2024_8.parquet'],
 '01190H': ['GLM_2024_8.parquet'],
 '01019H': ['GLM_2024_8.parquet'],
 '01089F': ['GLM_2024_8.parquet'],
 '01035H': ['GLM_2024_8.parquet'],
 '01057F': ['GLM_2024_8.parquet'],
 '01088F': ['GLM_2024_8.parquet'],
 '01080H': ['GLM_2024_8.parquet'],
 '01350F': ['GLM_2024_8.parquet'],
 '01023F': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01093H': ['GLM_2024_8.parquet'],
 '01085F': ['GLM_2024_8.parquet'],
 '01155H': ['GLM_2024_8.parquet'],
 '01015H': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01046H': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01170H': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01248H': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01178F': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01279H': ['GLM_2024_8.parquet', 'LI_2024_8.parquet'],
 '01302F': ['

Write the earthcare-id -> file dictionary as a parquet file.

In [ ]:
df = pd.Series(earthcare_ids).reset_index()
df.columns = ['earthcare_id', 'files']
df.to_parquet('/Volumes/T7/data/blanka/earthcare_id_mapping.parquet', engine='pyarrow', compression='zstd')

### Collection 2

In [ ]:
def process_col2_nc_to_geoparquet(file_list):
    """
    Reads EarthCARE NC files, expands 2D variables (cpr, subcluster_id) 
    into a flat dataframe, sorts, and saves as GeoParquet.
    """
    dataframes = []
    
    for file_path in file_list:
        try:
            with xr.open_dataset(file_path) as ds:

                df = ds.to_dataframe().reset_index()
                path_obj = Path(file_path)
                earthcare_id = path_obj.stem.split('_')[-1]
                df['earthcare_id'] = earthcare_id
                dataframes.append(df)

        except Exception as e:
            print(f"Skipping {file_path}: {e}")


    print("Concatenating all files...")
    full_df = pd.concat(dataframes, ignore_index=True)

    del dataframes
    gc.collect()

    print("Creating GeoDataFrame...")
    geometry = gpd.points_from_xy(full_df['longitude'], full_df['latitude'])
    full_df.drop(columns=['latitude', 'longitude', 'cpr'], inplace=True)
    gdf = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")

    del full_df
    gc.collect()

    print("Sorting data...")
    gdf.sort_values(
        by=['geometry', 'time'], 
        inplace=True, 
        ignore_index=True
    )

    return gdf

Process LI files in one go.

In [21]:
output_dir = '/Volumes/T7/data/blanka/C2_parquet/'
source = 'LI'
source_files = [f for f in files2 if source in f]

gdf = process_col2_nc_to_geoparquet(source_files)

gdf.to_parquet(
    output_dir + f'{source}_C2.parquet',
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)


Processing 2083 files...
Concatenating all files...
Creating GeoDataFrame...
Sorting data...


Process GLM files in one go.

In [22]:
output_dir = '/Volumes/T7/data/blanka/C2_parquet/'
source = 'GLM'
source_files = [f for f in files2 if source in f]

gdf = process_col2_nc_to_geoparquet(source_files)

gdf.to_parquet(
    output_dir + f'{source}_C2.parquet',
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)

Processing 2191 files...
Concatenating all files...
Creating GeoDataFrame...
Sorting data...


### Collection 3

Process the collection 3 files.

In [ ]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import timedelta

def transform_json_to_parquet(json_file_path):

    with open(json_file_path, 'r') as f:
        content = json.load(f)
    
    data = content['data']

    df_summary = pd.DataFrame(data).drop(
        columns=['minute_counts', 'minute_mean_lat', 'minute_mean_lon']
    )
    
    # Convert types
    df_summary['peak_datetime'] = pd.to_datetime(df_summary['peak_datetime'])
    gdf_summary = gpd.GeoDataFrame(
        df_summary,
        geometry=gpd.points_from_xy(df_summary['peak_lon'], df_summary['peak_lat']),
        crs="EPSG:4326"
    )
    
    # the Evolution GeoDataFrame (Detail) ---
    evolution_rows = []
    
    for entry in data:
        common_ids = {
            'orbit_frame': entry['orbit_frame'],
            'parent_cluster_id': entry['parent_cluster_id'],
            'cluster_id': entry['cluster_id'],
            'peak_datetime': pd.to_datetime(entry['peak_datetime'])
        }
        
        minutes = entry['minute_counts'].keys()
        
        for m_str in minutes:
            minute_offset = int(m_str)
            abs_time = common_ids['peak_datetime'] + timedelta(minutes=minute_offset)
        
            row = common_ids.copy()
            row['minute_offset'] = minute_offset
            row['absolute_time'] = abs_time
            row['li_count'] = entry['minute_counts'][m_str]
            row['latitude'] = entry['minute_mean_lat'][m_str]
            row['longitude'] = entry['minute_mean_lon'][m_str]
            
            evolution_rows.append(row)
            
    df_evolution = pd.DataFrame(evolution_rows)
    
    # Create Geometry (Track Location)
    gdf_evolution = gpd.GeoDataFrame(
        df_evolution,
        geometry=gpd.points_from_xy(df_evolution['longitude'], df_evolution['latitude']),
        crs="EPSG:4326"
    )
    
    gdf_evolution = gdf_evolution.sort_values(by=['parent_cluster_id', 'cluster_id', 'absolute_time', ])

    return gdf_summary, gdf_evolution

In [34]:
json_file_path = '/Volumes/T7/data/blanka/C3_EarthCARE_lightning_storm_catalogue.json'

gdf_summary, gdf_evolution = transform_json_to_parquet(json_file_path)

In [35]:
gdf_summary.to_parquet('/Volumes/T7/data/blanka/C3_parquet/gdf_summary.pq', index=False)
gdf_evolution.to_parquet('/Volumes/T7/data/blanka/C3_parquet/gdf_evolution.pq', index=False)

# Tests and Asserts


In [ ]:

import glob
import xarray as xr
import geopandas as gpd
import numpy as np
import os
import re

files = sorted(glob.glob('/Volumes/T7/data/blanka/C1/*.nc'))

In [37]:
files

['/Volumes/T7/data/blanka/C1/GLM_20240801T041007Z_20240801T051832Z_33_0_01000B.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T052943Z_20240801T074235Z_60975_423_01001A.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T054120Z_20240801T075418Z_48011_16_01001B.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T070219Z_20240801T091519Z_298341_173_01002A.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T115405Z_20240801T135135Z_276_0_01005A.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T132449Z_20240801T153658Z_5447_0_01006B.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T153508Z_20240801T172844Z_71_0_01007E.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T162830Z_20240801T175403Z_69_0_01007F.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T170444Z_20240801T191707Z_19282_77_01008E.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T182521Z_20240801T203818Z_62850_0_01009D.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T183710Z_20240801T204957Z_51240_41_01009E.nc',
 '/Volumes/T7/data/blanka/C1/GLM_20240801T19580

In [ ]:
def extract_file_info(file_path):
    filename = os.path.basename(file_path)
    parts = filename.split('_')
    source = parts[0]
    date_str = parts[1]
    year = int(date_str[:4])
    month = int(date_str[4:6])
    return source, year, month

np.random.seed(42)
rf = np.random.choice(files)  # Randomly select a file for testing
t, y, m = extract_file_info(rf)
t,y,m

('LI', 2025, 1)

In [39]:
new = gpd.read_parquet(f'/Volumes/T7/data/blanka/C1_parquet/{t}_{y}_{m}.parquet')
new

,group_id,group_time,latitude,longitude,radiance,flash_id,parallax_corrected_lat,parallax_corrected_lon,ec_time_diff,parent_cluster_id,cluster_id,distance_from_nadir,earthcare_id,geometry
0,273053035,2025-01-08 07:38:16.413699968,-54.901802,-71.552704,41.0,38701535,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03490H,POINT (-71.5527 -54.9018)
1,273053033,2025-01-08 07:38:16.411699968,-54.915302,-71.547302,26.0,38701535,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03490H,POINT (-71.5473 -54.9153)
2,273053034,2025-01-08 07:38:16.412700032,-54.885601,-71.460899,55.0,38701535,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03490H,POINT (-71.4609 -54.8856)
3,272771951,2025-01-08 07:04:53.109299968,-54.623703,-71.439301,5.0,38571823,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03490H,POINT (-71.4393 -54.6237)
4,273071260,2025-01-08 07:40:36.372400000,-54.580502,-70.980301,5.5,38710401,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03490H,POINT (-70.9803 -54.5805)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14453971,601187,2025-01-20 12:03:35.710300032,-58.746601,18.665100,9.0,215473,-58.617088,18.566853,108790406016,-1,NaN,NaN,03680F,POINT (18.6651 -58.7466)
14453972,223186241,2025-01-06 11:06:01.514499968,-55.088100,24.780602,21.0,26807039,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03462F,POINT (24.7806 -55.0881)
14453973,223186242,2025-01-06 11:06:01.514499968,-55.166401,24.429600,14.5,26807039,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03462F,POINT (24.4296 -55.1664)
14453974,223186240,2025-01-06 11:06:01.514499968,-55.036800,25.031700,10.5,26807039,NaN,NaN,-9223372036854775808,-1,NaN,NaN,03462F,POINT (25.0317 -55.0368)


In [ ]:
orig = xr.open_dataset(rf, decode_timedelta=True)
orig

<xarray.Dataset> Size: 2MB
Dimensions:                 (groups: 41268)
Dimensions without coordinates: groups
Data variables:
    group_id                (groups) uint64 330kB ...
    group_time              (groups) datetime64[ns] 330kB ...
    latitude                (groups) float32 165kB ...
    longitude               (groups) float32 165kB ...
    radiance                (groups) float32 165kB ...
    flash_id                (groups) uint32 165kB ...
    parallax_corrected_lat  (groups) float32 165kB ...
    parallax_corrected_lon  (groups) float32 165kB ...
    ec_time_diff            (groups) int64 330kB ...
    parent_cluster_id       (groups) int16 83kB ...
    cluster_id              (groups) float32 165kB ...
    distance_from_nadir     (groups) float32 165kB ...
Attributes: (12/13)
    Conventions:          CF-1.8
    title:                MTG-LI to EarthCARE collocated data
    summary:              Subset of MTG-LI L2 group product, collocated to Ea...
    institution:          European Space Agency (ESA)
    history:              Created on 2026-01-08
    source:               EarthCARE MSI/CPR + EUMETSAT MTG Lightning Imager (...
    ...                   ...
    geospatial_lat_min:   -14.45580005645752
    geospatial_lat_max:   3.6288001537323
    geospatial_lon_min:   4.619699954986572
    geospatial_lon_max:   9.250200271606445
    time_coverage_start:  2025-01-18T12:20:04Z
    time_coverage_end:    2025-01-18T14:39:57Z

In [42]:
for d in orig.data_vars:

    assert orig[d].dropna(dim='groups').isin(new[d].values).all().values

In [43]:
new[new.group_id == orig['group_id'].values[0]]

,group_id,group_time,latitude,longitude,radiance,flash_id,parallax_corrected_lat,parallax_corrected_lon,ec_time_diff,parent_cluster_id,cluster_id,distance_from_nadir,earthcare_id,geometry
5157450,93530770,2025-01-18 14:30:00.817600,2.3787,8.9505,11.5,14136553,NaN,NaN,-9223372036854775808,10,NaN,NaN,03650E,POINT (8.9505 2.3787)


In [53]:
subset = new[new.group_id.isin(orig['group_id'].values) & new.group_time.isin(orig['group_time'].values)]
subset = subset.sort_values(by='group_id').reset_index(drop=True)
orig = orig.sortby("group_id")


columns_to_check = ['group_id','latitude', 'longitude', 'radiance', 'flash_id', 'parent_cluster_id', 'cluster_id', 'distance_from_nadir'] #  'group_time', 
for c in columns_to_check:
    assert np.isclose(subset[c], orig[c].values, equal_nan=True).all()